# Shotpath — Ball + Rim Detector

Fine-tunes a YOLO-nano object detector to find the ball and rim in sideline free-throw video, then exports a quantized TFLite model for on-device inference via `react-native-fast-tflite`.

**Run this in Google Colab with a T4 GPU runtime** (Runtime → Change runtime type → GPU).

Pipeline:
1. Install ultralytics + roboflow.
2. Pull a labeled basketball dataset from Roboflow Universe.
3. Fine-tune `yolo11n.pt` on the dataset.
4. Evaluate on the held-out val split.
5. Export to TFLite with int8 quantization.
6. Download the `.tflite` file for the app.

**Target model size:** <10 MB. **Target mAP50:** >0.7 on the Roboflow val split before we trust it on our own clips.

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

## 2. Pull dataset from Roboflow

Go to https://universe.roboflow.com and find a basketball detection dataset with **ball** and **rim** (or **hoop**) classes. Good starting points to search:

- `basketball detection` — several workspaces have multi-class annotations (ball, rim, player).
- `basketball hoop` — narrower datasets focused on rim.

Once you pick one, click **Download Dataset → YOLOv11** and it shows you a snippet like the one below. Paste your own API key, workspace, project, and version number.

Our minimum bar: at least 500 labeled images, both classes present, sideline or near-sideline views represented.

In [ ]:
from roboflow import Roboflow
import os

# Replace these with your own Roboflow project coordinates.
ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', 'PASTE_YOUR_KEY_HERE')
WORKSPACE = 'PASTE_WORKSPACE'
PROJECT = 'PASTE_PROJECT'
VERSION = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download('yolov11')
print('Dataset at:', dataset.location)

## 3. Fine-tune YOLOv11n

YOLOv11n is the smallest variant (~2.6M params). A 50-epoch run on ~2k images takes ~15 min on a T4. Increase `epochs` if mAP is still climbing at the end.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')  # pretrained on COCO
results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    project='shotpath-ball-rim',
    name='run1',
    patience=15,
)
print('Best weights:', results.save_dir)

## 4. Evaluate on the val split

In [ ]:
best = YOLO(f'{results.save_dir}/weights/best.pt')
metrics = best.val()
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)
print('Per-class mAP50:', metrics.box.maps)

## 5. Quick visual sanity check on your own clip

Upload one of your own free-throw frames (PNG or JPG) via the Colab file panel, then run this cell. If the ball and rim boxes don't land on the right objects, the dataset isn't generalizing and you need more diverse training data before trusting the export.

In [ ]:
# Replace with the path to an uploaded sample frame.
SAMPLE_FRAME = '/content/sample_sideline_frame.jpg'
preds = best.predict(SAMPLE_FRAME, save=True, conf=0.25)
for p in preds:
    print(p.boxes)

## 6. Export to quantized TFLite

Int8 quantization shrinks the model ~4x with a small accuracy cost. Ultralytics will emit `best_saved_model/best_int8.tflite` or similar. Confirm file size under 10 MB.

In [ ]:
export_path = best.export(format='tflite', int8=True, imgsz=640)
print('Exported to:', export_path)

import os
size_mb = os.path.getsize(export_path) / (1024 * 1024)
print(f'Model size: {size_mb:.2f} MB')
assert size_mb < 10, 'Model too large — reduce imgsz or switch to yolo11n from a larger variant.'

## 7. Download the model

Save the `.tflite` locally, then drop it into the repo at `assets/models/ball_rim.tflite`. Phase 2 wiring will load it via `react-native-fast-tflite`.

In [ ]:
from google.colab import files
files.download(export_path)